# Risk labels on our forecasts

Applies the risk flags of [`risk-definition.ipynb`](risk-definition.ipynb) (thresholds, day rules, limits of the label) to the test-window forecasts of spec 09's picks ([`regression-visualization-best-models.ipynb`](../05_modeling/regression-visualization-best-models.ipynb)) and of SMARD, and scores them against the actual labels; forecast method and caveats are in [`regression-models-claude.ipynb`](../05_modeling/regression-models-claude.ipynb).
Reads `data/risk_classification/risk_labels_{daily,hourly}.csv`, `data/models/model_{forecast_errors_hourly,scoreboard}.csv` and `data/metrics/smard_forecast_errors_hourly.csv`; writes `data/risk_classification/model_risk_labels_{daily,hourly}.csv`.
*The label is a national-balance proxy, not a validated intervention record.*

## 1 Setup and checks

### 1.1 Settings

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

PICKS = {"high": "xgb_hybrid", "low": "random_forest_hybrid"}  # spec 09 rank 1: high_extreme / low_extreme
PICK_CATEGORIES = {"high": ["high_extreme"], "low": ["low_extreme", "below_zero"]}  # checked in 1.3
SPLIT = "rolling"            # split method of the picks (spec 09's CANDIDATE_SPLIT)
BASES = {"high": ["rolling"], "low": ["rolling", "zero"]}
RULES = ["any", "3h"]
MIN_BIN_SHARE = 0.5          # spec 09's forecast-share floor, used by the pick check
ZOOM_WEEKS = {"high": [None, None], "low": [None, None]}  # a Monday "YYYY-MM-DD" overrides a zoom rule
EXPORT_ENABLED = True
MARGIN_WINDOW = 10_000       # MWh: axis range of the day-margin plot around 0

# Day rules as durations, as in risk-definition.ipynb §3.1 (RESOLUTION is measured in §2.1)
PERSISTENCE = pd.Timedelta("3h")
DAY_COMPLETENESS = 23 / 24

# Label and colour per row, copied from regression-visualization-best-models.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
}

# Thresholds, flagged hours and hit cells per (direction, basis), never a model's colour
TAIL_COLOR = {
    ("high", "rolling"): "#E6B800",  # gold, spec 09's high_extreme
    ("low", "rolling"): "#17BECF",   # cyan, low_extreme
    ("low", "zero"): "#9EDAE5",      # light cyan, below_zero
}

# Day and hour outcomes; a hit takes the TAIL_COLOR, flagged outcomes are edged in it
OUTCOME_COLOR = {
    "miss": "#1C1C1C",
    "false alarm": "#AEB8C5",
    "quiet": "#F2F4F7",
    "not evaluable": "white",  # also days outside the test window
}


def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def style_plain(ax, title, ylabel, xlabel):
    """The same look for plots without a time axis."""
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel(xlabel, color="grey")
    ax.set_ylabel(ylabel, color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

### 1.2 Loading

In [ ]:
# data/: first in the working directory, then in each parent
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")

LABELS_SOURCE = "notebooks/03_risk_classification/risk-definition.ipynb"
MODEL_SOURCE = "notebooks/05_modeling/regression-models-claude.ipynb with EXPORT_ENABLED = True"
SMARD_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SOURCES = {
    "labels_daily": ("risk_classification/risk_labels_daily.csv", LABELS_SOURCE),
    "labels_hourly": ("risk_classification/risk_labels_hourly.csv", LABELS_SOURCE),
    "hourly": ("models/model_forecast_errors_hourly.csv", MODEL_SOURCE),
    "scoreboard": ("models/model_scoreboard.csv", MODEL_SOURCE),
    "smard": ("metrics/smard_forecast_errors_hourly.csv", SMARD_SOURCE),
}
STAMP = "%Y-%m-%d %H:%M:%S"


def load(name):
    """One file as a DataFrame; stops if it is missing."""
    path, producer = SOURCES[name]
    path = DATA_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {producer}.")
    frame = pd.read_csv(path)
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format=STAMP)
    if "date" in frame.columns:
        frame["date"] = pd.to_datetime(frame["date"], format="%Y-%m-%d")
    return frame


# Label columns stay three-state: an empty cell is "not evaluable", never False
LABEL_PAIRS = [(direction, basis) for direction in BASES for basis in BASES[direction]]
DAILY_THRESHOLDS = [f"{d}_threshold_{b}" for d, b in LABEL_PAIRS]
DAILY_FLAGS = [f"{d}_risk_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES]
DAILY_RANGES = [f"{d}_range_{e}_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES for e in ("start", "end")]
HOURLY_FLAGS = [f"{d}_risk_hour_{b}" for d, b in LABEL_PAIRS]

labels_daily = load("labels_daily").set_index("date")[DAILY_THRESHOLDS + DAILY_FLAGS + DAILY_RANGES]
labels_daily[DAILY_FLAGS] = labels_daily[DAILY_FLAGS].astype("boolean")
for column in DAILY_RANGES:
    labels_daily[column] = pd.to_datetime(labels_daily[column], format=STAMP)
labels_hourly = load("labels_hourly").set_index("timestamp")[["residual_load"] + HOURLY_FLAGS]
labels_hourly[HOURLY_FLAGS] = labels_hourly[HOURLY_FLAGS].astype("boolean")

hourly = load("hourly")
scoreboard = load("scoreboard")
smard = load("smard").set_index("timestamp")

# Test window: the hours and local dates of the model export
TEST_HOURS = pd.DatetimeIndex(sorted(hourly["timestamp"].unique()))
TEST_DAYS = pd.DatetimeIndex(sorted(TEST_HOURS.normalize().unique()))

# The picks' forecasts; a missing pick stays an empty column and fails the pick check
picked = hourly[hourly["split_method"].eq(SPLIT) & hourly["model"].isin(PICKS.values())]
FORECAST = picked.pivot(index="timestamp", columns="model", values="forecast").reindex(
    index=TEST_HOURS, columns=list(dict.fromkeys(PICKS.values()))
)
ACTUAL = picked.groupby("timestamp")["residual_load"].first().reindex(TEST_HOURS)
SMARD_FORECAST = smard["fc_residual_load"].reindex(TEST_HOURS)

# Common hours: an actual, SMARD's forecast and a forecast from both picks
COMMON = TEST_HOURS[ACTUAL.notna() & SMARD_FORECAST.notna() & FORECAST.notna().all(axis=1)]

print(f"test window : {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d} ({len(TEST_DAYS)} days, {len(TEST_HOURS):,} hours)")
print(f"picks       : high {PICKS['high']}, low {PICKS['low']} ({SPLIT} split)")
print(f"common hours: {len(COMMON):,} of {len(TEST_HOURS):,}")

### 1.3 Pick check

Each pick needs a `SPLIT` row in both model exports and must pass spec 09's bin rule in every `PICK_CATEGORIES` category. On a failure the notebook stops; after spec 09 is re-run with new picks, edit `PICKS`.

In [ ]:
SMARD_ROW = ("smard", "none")
VALUE = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="count")
HOURLY_ROWS = set(zip(hourly["model"], hourly["split_method"]))


def failed_rules(row, category):
    """Spec 09's bin rule: the rules a row fails in a category; empty if it qualifies."""
    failed = []
    for view in ["actual", "forecast"]:
        metric = f"MAE_{category}_by_{view}"
        if not VALUE.loc[row, metric] < VALUE.loc[SMARD_ROW, metric]:
            failed.append(f"{metric} {VALUE.loc[row, metric]:,.0f} not below SMARD's {VALUE.loc[SMARD_ROW, metric]:,.0f}")
    by_actual = COUNT.loc[row, f"MAE_{category}_by_actual"]
    share = COUNT.loc[row, f"MAE_{category}_by_forecast"] / by_actual if by_actual else np.nan
    if not share >= MIN_BIN_SHARE:
        failed.append(f"forecast share {share:.2f} < {MIN_BIN_SHARE}")
    return failed


problems = []
for direction, model in PICKS.items():
    row = (model, SPLIT)
    if row not in HOURLY_ROWS:
        problems.append(f"{direction} {model}: no {SPLIT} row in model_forecast_errors_hourly.csv")
    if row not in VALUE.index:
        problems.append(f"{direction} {model}: no {SPLIT} row in model_scoreboard.csv")
        continue
    for category in PICK_CATEGORIES[direction]:
        failed = failed_rules(row, category)
        problems += [f"{direction} {model} in {category}: {rule}" for rule in failed]
        by_actual, by_forecast = (f"MAE_{category}_by_{view}" for view in ["actual", "forecast"])
        print(
            f"{direction:<4} {model:<20} {category:<12} "
            f"MAE by actual {VALUE.loc[row, by_actual]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_actual]:,.0f}), "
            f"by forecast {VALUE.loc[row, by_forecast]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_forecast]:,.0f}), "
            f"forecast share {COUNT.loc[row, by_forecast] / COUNT.loc[row, by_actual]:.2f}: "
            f"{'fails' if failed else 'passes'}"
        )

if problems:
    raise RuntimeError(
        f"pick check failed for PICKS = {PICKS}:\n  " + "\n  ".join(problems)
        + "\nRe-run regression-visualization-best-models.ipynb and edit PICKS; no model is picked here."
    )
print("pick check passed")

### 1.4 Snapshot self-check

Labels, model exports and SMARD file must come from one `smard.csv` fetch; the notebook never joins on a partial overlap.

In [ ]:
REGENERATE = "API-connection → risk-definition → forecast-metrics-claude → regression-models-claude"


def require(ok, problem):
    """Stop the notebook, naming the regeneration order."""
    if not ok:
        raise RuntimeError(f"self-check failed: {problem}. Regenerate in this order: {REGENERATE}.")


require(len(COMMON) > 0, "no common hours")

# Every test day and hour has a label row
missing_days = TEST_DAYS.difference(labels_daily.index)
require(missing_days.empty, f"{len(missing_days)} test days missing from risk_labels_daily.csv")
missing_hours = TEST_HOURS.difference(labels_hourly.index)
require(missing_hours.empty, f"{len(missing_hours)} test hours missing from risk_labels_hourly.csv")

# Same actuals in the three hourly files
for name, other in [
    ("smard_forecast_errors_hourly.csv", smard["residual_load"]),
    ("risk_labels_hourly.csv", labels_hourly["residual_load"]),
]:
    gap = float((ACTUAL[COMMON] - other.reindex(COMMON)).abs().max())
    require(gap <= 1e-6, f"residual_load of the model export and {name} differ by up to {gap:,.2f} MWh")

print(
    f"self-check passed: labels, model exports and SMARD file share one snapshot "
    f"({len(TEST_DAYS)} days, {len(COMMON):,} common hours)"
)